# AI Nutrition Labels for AI Agents — POC (v0.3-draft)

This notebook demonstrates the end-to-end flow:

1. **Schema** — a standardized, domain-agnostic JSON Schema (v0.3-draft) mirroring the original ["AI Nutrition Labels for Everyone"](https://medium.com/@aashkafirst/ai-nutrition-labels-for-everyone-simplifying-model-cards-from-geek-to-street-7dec406b56f3) framework: Model Identity, Functional Capabilities, Performance Value (PV), Safety Value (SV), Bias Value (BV), Environmental Impact, Privacy Seal, and Limitations.
2. **Registry** — 7 **real** models (Claude Sonnet 4.6, GPT-4o, Gemini 1.5 Pro, DALL-E 3, Stable Diffusion 3.5 Large, NLLB-200, Sarvam-Translate), labeled from real public sources where they exist -- see each label's `estimated_fields` list for what's a flagged placeholder.
3. **Real energy benchmark** — environmental figures are now tied to the real [AI Energy Score](https://huggingface.co/spaces/AIEnergyScore/Leaderboard) benchmark and methodology, plus real provider-published energy figures (Google's Gemini report, OpenAI's stated ChatGPT energy use) and the Luccioni et al. image-generation energy study, wherever available.
4. **REST API** — FastAPI service exposing `/labels`, `/labels/search`, `/labels/{id}`.
5. **Use case 1 (procurement)** — an agent assembling a model stack for a **maternal health app in rural India**, reasoning from the standardized global fields. Includes a real head-to-head: **Sarvam-Translate** (India-built, 4B params) vs. **NLLB-200** (Meta, 54.5B params) for the translation slot.
6. **Use case 2 (classroom safety snapshot)** — a **teacher in Nairobi** deciding which conversational AI is safe for her students, using real [KORA child-safety benchmark](https://korabench.ai/) data attached under each label's optional `extensions` object, rendered as a short, non-overwhelming snapshot instead of the full label.

## 1. Compute composite scores, then validate against the JSON Schema

`seed_source.json` holds only raw component data (some real, some estimated).
`build_seed_data.py` derives `performance_value` (PV), `safety_value` (SV),
`bias_value` (BV), and the environmental grades from it, then
`seed_data.json` is validated against the schema.

In [1]:
import json, sys, subprocess, pathlib

PROJECT_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebook" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

subprocess.run([sys.executable, "registry/build_seed_data.py"], cwd=PROJECT_ROOT, check=True)

import jsonschema

schema = json.loads((PROJECT_ROOT / "schema" / "ai_nutrition_label.schema.json").read_text())
labels = json.loads((PROJECT_ROOT / "registry" / "seed_data.json").read_text())

for label in labels:
    jsonschema.validate(instance=label, schema=schema)

print(f"\nAll {len(labels)} labels validate against schema v{schema['version']}")

Computed 7 labels -> /Users/aashka/Documents/Claude/AI Nutrition Labels For AI Agents POC/registry/seed_data.json
  Claude Sonnet 4.6            PV=86.5   SV=66.33  BV=N/A    Carbon=D   Energy=**    Water=Low
  GPT-4o                       PV=84.38  SV=66.0   BV=N/A    Carbon=B   Energy=****  Water=Water Saver
  Gemini 1.5 Pro               PV=78.97  SV=65.67  BV=N/A    Carbon=A   Energy=***** Water=Water Saver
  DALL-E 3                     PV=50.75  SV=66.67  BV=N/A    Carbon=D   Energy=**    Water=Low
  Stable Diffusion 3.5 Large   PV=46.5   SV=57.67  BV=N/A    Carbon=D   Energy=*     Water=Low
  NLLB-200 (54.5B MoE)         PV=39.0   SV=N/A    BV=0.73   Carbon=C   Energy=****  Water=Water Saver
  Sarvam-Translate             PV=42.0   SV=N/A    BV=N/A    Carbon=A+  Energy=***** Water=Water Saver

All 7 labels validate against schema v0.3-draft


## 2. Real vs. estimated: the AI Energy Score integration

None of these 7 models are actually measured on the official [AI Energy Score leaderboard](https://huggingface.co/spaces/AIEnergyScore/Leaderboard) -- it only benchmarks self-hostable models on standardized GPU hardware, so closed APIs (Claude, GPT-4o, Gemini, DALL-E 3) can never appear on it, and even the open-weight models here (Stable Diffusion 3.5 Large, NLLB-200, Sarvam-Translate) aren't in its current snapshot. Where a provider has separately published a real per-query energy figure (Google's Gemini report, OpenAI's stated ChatGPT figure), we use it directly. Otherwise every number is an estimate extrapolated from real comparable data points -- and every label says so explicitly in `environmental_impact.ai_energy_score.disclosure`.

In [2]:
for label in labels:
    env = label["environmental_impact"]
    aes = env["ai_energy_score"]
    print(f"{label['model_identity']['name']:<28} {env['estimated_energy_wh_per_query']:>6} Wh/query  "
          f"Carbon={env['carbon_footprint_grade']:<3} on_official_leaderboard={aes['on_official_leaderboard']}")
print("\nSample disclosure (Gemini 1.5 Pro -- our best-grounded entry):")
print(next(l for l in labels if l["label_id"] == "gemini-1.5-pro-002")["environmental_impact"]["ai_energy_score"]["disclosure"])

Claude Sonnet 4.6               2.8 Wh/query  Carbon=D   on_official_leaderboard=False
GPT-4o                         0.34 Wh/query  Carbon=B   on_official_leaderboard=False
Gemini 1.5 Pro                 0.24 Wh/query  Carbon=A   on_official_leaderboard=False
DALL-E 3                        2.9 Wh/query  Carbon=D   on_official_leaderboard=False
Stable Diffusion 3.5 Large      3.5 Wh/query  Carbon=D   on_official_leaderboard=False
NLLB-200 (54.5B MoE)            0.7 Wh/query  Carbon=C   on_official_leaderboard=False
Sarvam-Translate              0.011 Wh/query  Carbon=A+  on_official_leaderboard=False

Sample disclosure (Gemini 1.5 Pro -- our best-grounded entry):
Gemini 1.5 Pro is a closed proprietary API and cannot be run on AI Energy Score's standardized self-hosted GPU harness, so it is not on the official leaderboard. The 0.24 Wh/query figure is real (Google-published, see methodology_note) rather than a placeholder, making this registry's best-grounded energy figure -- but the st

## 3. Build the SQLite registry and start the REST API

In [3]:
subprocess.run([sys.executable, "registry/init_db.py"], cwd=PROJECT_ROOT, check=True)

Loaded 7 labels into /Users/aashka/Documents/Claude/AI Nutrition Labels For AI Agents POC/registry/registry.db


CompletedProcess(args=['/Users/aashka/Documents/Claude/AI Nutrition Labels For AI Agents POC/.venv/bin/python3', 'registry/init_db.py'], returncode=0)

In [4]:
import time, requests

API_BASE = "http://127.0.0.1:8000"

def api_is_up():
    try:
        return requests.get(f"{API_BASE}/labels", timeout=1).status_code == 200
    except requests.exceptions.ConnectionError:
        return False

server_proc = None
if not api_is_up():
    server_proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "api.main:app", "--port", "8000"],
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for _ in range(20):
        if api_is_up():
            break
        time.sleep(0.5)

print("API up:", api_is_up())

API up: True


## 4. Query the registry over REST

In [5]:
resp = requests.get(f"{API_BASE}/labels")
for row in resp.json():
    print(f"{row['name']:<28} PV={row['performance_value']:<6} Carbon={row['carbon_footprint_grade']:<3} "
          f"OpenWeights={bool(row['open_weights'])} Privacy={row['privacy_seal']:<7} "
          f"ChildSafety={row['child_safety_score_pct']}")

Claude Sonnet 4.6            PV=86.5   Carbon=D   OpenWeights=False Privacy=Gold    ChildSafety=72.0
GPT-4o                       PV=84.38  Carbon=B   OpenWeights=False Privacy=Silver  ChildSafety=37.0
Gemini 1.5 Pro               PV=78.97  Carbon=A   OpenWeights=False Privacy=Silver  ChildSafety=45.0
DALL-E 3                     PV=50.75  Carbon=D   OpenWeights=False Privacy=Silver  ChildSafety=None
Stable Diffusion 3.5 Large   PV=46.5   Carbon=D   OpenWeights=True Privacy=Gold    ChildSafety=None
NLLB-200 (54.5B MoE)         PV=39.0   Carbon=C   OpenWeights=True Privacy=Gold    ChildSafety=None
Sarvam-Translate             PV=42.0   Carbon=A+  OpenWeights=True Privacy=Gold    ChildSafety=None


## 5. Use case 1 — maternal health app procurement for rural India

The agent applies **different constraints per component**, because each
differs in where patient data physically goes: a cloud LLM is acceptable
(a health worker's phone sometimes has connectivity), but the vision triage
and translation models must run fully **on-device** (`open_weights`, a real
structural property, standing in for a fabricated `offline_capable` flag).
The environmental grade -- now backed by real provider data -- turns out to
be a genuine tiebreaker: Gemini 1.5 Pro's real, Google-published efficiency
figures win it the LLM slot over Claude Sonnet 4.6.

For translation, both NLLB-200 and Sarvam-Translate are open-weight and
support Hindi/Marathi, so this one comes down to score rather than a hard
reject -- **Sarvam-Translate wins**: it's a much smaller (4B vs. 54.5B),
India-built model fine-tuned specifically for the 22 official Indian
languages, with a real published human-evaluation finding that it beats
much larger general-purpose models on Indian-language translation quality,
plus a real efficiency edge that earns it an A+ carbon grade vs. NLLB's C.

In [6]:
from agent.procurement_agent import run_procurement

decision = run_procurement()


=== Selecting MULTIMODAL model ===
  ELIGIBLE  Claude Sonnet 4.6           
  ELIGIBLE  GPT-4o                      
  ELIGIBLE  Gemini 1.5 Pro              
  Comparing eligible candidates metric-by-metric (no composite score):
    Compare on healthbench_score: {'Claude Sonnet 4.6': '38 (estimated)', 'GPT-4o': '32 (real)', 'Gemini 1.5 Pro': '44 (estimated)'}
      -> eliminates ['Claude Sonnet 4.6', 'GPT-4o']; ['Gemini 1.5 Pro'] remain
  SELECTED  Gemini 1.5 Pro
  Caution (Gemini 1.5 Pro provider disclaims): ['medical diagnosis'] -- deploy with mandatory human review for these uses.

=== Selecting IMAGE model ===
  REJECTED  DALL-E 3                     -> not open-weight -- cannot be self-hosted on-device, so it can't run in a rural clinic with unreliable connectivity (this deployment's hard requirement)
  ELIGIBLE  Stable Diffusion 3.5 Large  
  Only one eligible candidate -- no comparison needed: Stable Diffusion 3.5 Large
  SELECTED  Stable Diffusion 3.5 Large
  Caution (Stable D

## 6. Use case 2 — a teacher in Nairobi choosing a classroom-safe AI model

Different problem, different shape of answer. The teacher doesn't need PV,
context window size, or GPU energy figures -- she needs to know: is this
safe for my students, what should I watch out for, and does it keep their
data private. `agent/teacher_snapshot.py` filters the full label down to
exactly that, using real [KORA](https://korabench.ai/) child-safety data
attached under each label's `extensions.child_safety` -- the domain-specific
overlay slot the schema reserved for exactly this kind of use case.

In [7]:
from agent.teacher_snapshot import recommend_for_classroom, print_snapshot

snapshot = recommend_for_classroom(min_kora_score=50)
if snapshot:
    print()
    print_snapshot(snapshot)

=== Classroom AI model recommendation (Nairobi teacher use case) ===

  Claude Sonnet 4.6    KORA score=72%  -> Usable with standard classroom supervision
  Gemini 1.5 Pro       KORA score=45%  -> Not recommended for direct student use without heavy adult oversight  (BELOW your minimum threshold)
  GPT-4o               KORA score=37%  -> Not recommended for direct student use without heavy adult oversight  (BELOW your minimum threshold)

  RECOMMENDED: Claude Sonnet 4.6


AI Nutrition Label -- Classroom Snapshot: Claude Sonnet 4.6 (Anthropic)
------------------------------------------------------------
Child safety (KORA benchmark): 72%  -- Usable with standard classroom supervision
  This score is our own estimate, not a number read directly off KORA's leaderboard -- see score_basis for how it was derived.
  KORA's own caveat: KORA itself cautions this is not a safety guarantee -- even the best-scoring models fail or are merely 'adequate' on a meaningful share of tested scenarios.

Pr

Notice how short that is compared to the full label below -- that's the point. Everything not needed for *this* decision (performance benchmarks, context window, GPU energy, carbon grade, provenance metadata) is simply not there.

In [8]:
full_label = next(l for l in labels if l["model_identity"]["name"] == snapshot["model_name"])
print(f"Full label has {len(json.dumps(full_label))} characters across "
      f"{len(full_label)} top-level sections.")
print(f"Teacher snapshot has {len(json.dumps(snapshot))} characters across "
      f"{len(snapshot)} top-level sections: {list(snapshot.keys())}")

Full label has 9359 characters across 12 top-level sections.
Teacher snapshot has 1171 characters across 5 top-level sections: ['model_name', 'provider', 'child_safety', 'privacy', 'watch_out_for']


## 7. Render the procurement decision as a full nutrition-label-style summary

In [9]:
def render_summary(role, label):
    if label is None:
        print(f"[{role}] NO MODEL SELECTED — escalated to human reviewer\n")
        return
    mi, perf, sb, env, priv = (label["model_identity"], label["performance"],
                                 label["safety_and_bias"], label["environmental_impact"], label["privacy"])
    print(f"[{role}] {mi['name']}  ({mi['manufacturer']}, {mi['release_date']})")
    print(f"   Performance Value (PV):  {perf['performance_value']}   [{perf['formula']}]")
    print(f"   Safety Value (SV):       {sb['safety_value']}   [{sb['safety_formula']}]")
    print(f"   Bias Value (BV):         {sb['bias_value']}   [{sb['bias_formula']}]")
    print(f"   Environmental:           Carbon {env['carbon_footprint_grade']} | "
          f"{'*' * env['energy_rating_stars']} energy | {env['green_energy_seal_pct']}% green | "
          f"water: {env['water_footprint_level']} | on AI Energy Score leaderboard: {env['ai_energy_score']['on_official_leaderboard']}")
    print(f"   Privacy Seal:            {priv['privacy_seal']}  (data used for training: {priv['data_used_for_training']})")
    print(f"   Open weights:            {mi['open_weights']}")
    print(f"   Not recommended for:     {label['limitations']['not_recommended_for']}")
    print(f"   ({len(label['estimated_fields'])} fields in this label are estimated placeholders, not sourced facts)")
    print()

for role, label in decision.items():
    render_summary(role, label)

[language_model] Gemini 1.5 Pro  (Google DeepMind, 2024-05-14)
   Performance Value (PV):  78.97   [PV = (GR + C + M + CSR) / 4]
   Safety Value (SV):       65.67   [SV = (Rtoxic + (100 - Rnontoxic) + (100 - IR)) / 3]
   Bias Value (BV):         None   [BV = weighted average of bias benchmark values (0-1 scale)]
   Environmental:           Carbon A | ***** energy | 100% green | water: Water Saver | on AI Energy Score leaderboard: False
   Privacy Seal:            Silver  (data used for training: True)
   Open weights:            False
   Not recommended for:     ['medical diagnosis', 'legal advice', 'financial advice']
   (13 fields in this label are estimated placeholders, not sourced facts)

[vision_model] Stable Diffusion 3.5 Large  (Stability AI, 2024-10-22)
   Performance Value (PV):  46.5   [PV = (GR + C + M + CSR) / 4]
   Safety Value (SV):       57.67   [SV = (Rtoxic + (100 - Rnontoxic) + (100 - IR)) / 3]
   Bias Value (BV):         None   [BV = weighted average of bias benchma

## 8. Cleanup (optional)

In [10]:
if server_proc is not None:
    server_proc.terminate()
    print("API server stopped.")
else:
    print("API server was started externally — leaving it running.")

API server stopped.
